# Week 5 Presentation Brief — Variation A
## ☀️ Solar Farm Economics: Optimal Panel Configuration
**SCIE1500**

> Work through all parts during the Week 5 lab. Your **10-minute Week 6 presentation** should cover: the problem, your model, optimal configuration, and climate sensitivity
> **What to submit:** every group member must individually upload their own copy of the same completed presentation slides (PDF or PowerPoint) to the LMS after your presentation — this lets your instructor verify who participated.


---
## 📋 Scenario

![Solar farm revenue optimisation curve](../images/W5A_solar_farm.svg)

A renewable energy company designs a 50-ha solar farm near Geraldton. Annual revenue (thousands of dollars) depends on panel row spacing $x$ (metres):

$$R(x) = -2x^2 + 180x - 1600$$

**Constraints:** $8 \leq x \leq 60$ metres

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Find and classify critical points of the revenue function | ~25 min |
| B | Identify constraint boundaries and find the break-even range | ~20 min |
| C | Conduct a climate sensitivity and upgrade analysis | ~15 min |
| D | Verify the critical point symbolically using SymPy | ~10 min |

In [ ]:
# Run first — loads libraries for this session
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Solar farm revenue model
def R(x):
    'Annual revenue (thousands $) for panel spacing x metres.'
    return -2*x**2 + 180*x - 1600

def dR(x):
    'Marginal revenue ($/m of spacing change).'
    return -4*x + 180

def d2R(x):
    'Second derivative.'
    return -4    # constant negative → always concave down → critical point is maximum

print("R(8)  =", R(8),  "($000)")
print("R(45) =", R(45), "($000)")
print("R(60) =", R(60), "($000)")

---
## Part A: Critical Point Analysis (~25 min)

In [ ]:
# A.1 — Find the critical point: solve R'(x) = 0
# -4x + 180 = 0  →  x = 45
x_crit = 180 / 4
R_crit = R(x_crit)
R2 = d2R(x_crit)

print(f"R'(x) = -4x + 180 = 0  →  x* = {x_crit} metres")
print(f"R''(x) = {R2}  (< 0 → maximum confirmed)")
print(f"R({x_crit:.0f}) = ${R_crit:,.0f} thousand/year")

Check feasibility and compare to boundaries.

In [ ]:
# A.2 — Check feasibility and compare to boundaries
for x_s, label in [(x_crit, "Optimal (x=45)"), (8, "Min spacing (x=8)"), (60, "Max spacing (x=60)")]:
    feasible = 8 <= x_s <= 60
    print(f"{label}: R = ${R(x_s):,.0f}k  feasible? {feasible}")

Plot revenue curve with constraint boundaries.

In [ ]:
# A.3 — Plot revenue curve with constraint boundaries
x_vals = np.linspace(5, 65, 300)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_vals, R(x_vals), "steelblue", lw=2, label="Revenue R(x)")
ax.plot(x_crit, R_crit, "ro", ms=12, label=f"Optimal: x={x_crit:.0f}m, R=${R_crit:.0f}k")
ax.axvline(8,  color="orange", ls="--", alpha=0.7, label="Min spacing (8m)")
ax.axvline(60, color="orange", ls="--", alpha=0.7, label="Max spacing (60m)")
ax.axhline(0,  color="k", lw=0.8)
ax.set_xlabel("Panel Row Spacing x (metres)")
ax.set_ylabel("Annual Revenue ($000)")
ax.set_title("Solar Farm Revenue Optimization")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(-500, 2500)
plt.tight_layout()
plt.show()

### Investment interpretation checkpoint

Use the curve to justify a configuration decision, not just to locate a maximum.

- Describe the revenue trade-off on either side of 45 m using the sign of $R'(x)$.
- Compare the optimum with the operational boundaries and explain why feasibility matters even when calculus gives one critical point.
- Identify one factor not represented by the baseline revenue curve that a company should test before committing capital, such as installation cost, maintenance, or a changed weather pattern.

✏️ **Interpreting the Revenue Peak:**
1. Why does revenue *decrease* both below and above the optimal spacing?
2. The unconstrained optimum $x^* = 45$ m falls inside $[8, 60]$. What if it fell outside?

```
Answers:
1. ...
2. ...
```

---
## Part B: Break-Even Analysis (~20 min)

In [ ]:
# B.1 — Break-even points: solve R(x) = 0
# -2x² + 180x - 1600 = 0  →  x² - 90x + 800 = 0
a, b, c = -2, 180, -1600
disc = b**2 - 4*a*c
x_be1 = (-b + np.sqrt(disc)) / (2*a)
x_be2 = (-b - np.sqrt(disc)) / (2*a)

if x_be1 > x_be2:
    x_be1, x_be2 = x_be2, x_be1

print(f"Break-even points: x = {x_be1:.1f} m and x = {x_be2:.1f} m")
print(f"Farm is profitable for x ∈ [{x_be1:.1f}, {x_be2:.1f}] metres")
print(f"Both within operational constraints [8, 60]? {8 <= x_be1 and x_be2 <= 60}")

---
## Part C: Climate Sensitivity (~15 min)

Two different what-if scenarios could affect the farm's revenue: **climate change** might reduce output (e.g. more cloud cover or heat-related efficiency losses), while a **high-efficiency panel upgrade** might increase it. We model both as simple percentage scalings of the baseline revenue curve $R(x)$ — a 15% cut for the climate scenario, a 20% lift for the upgrade — and compare the revenue each gives at the optimal spacing.

✏️ **Before running the cell below:** notice that the code reuses `x_opt = 45` for all three scenarios — the same spacing found in Part A, with no new optimisation step for either scenario. Is that a shortcut, or is it mathematically justified? Think about what multiplying $R(x)$ by a positive constant does to *where* its maximum occurs, before you run the cell and see the results.

In [ ]:
# C.1 — Revenue under two different scenarios

# Climate change: assume reduced solar output cuts revenue by 15%
def R_climate(x):
    'Revenue under climate change (−15%).'
    return 0.85 * R(x)

# Panel upgrade: assume higher-efficiency panels lift revenue by 20%
def R_upgrade(x):
    'Revenue with high-efficiency panels (+20%).'
    return 1.20 * R(x)

# Note: x_opt is reused unchanged across all three scenarios below --
# see the question above before reading into why.
for fn, label in [(R, "Baseline"), (R_climate, "Climate (−15%)"), (R_upgrade, "Upgrade (+20%)")]:
    x_opt = 45
    print(f"{label:>20}: optimal R = ${fn(x_opt):,.0f}k at x = {x_opt} m")

The chart below shows all three scenarios.

In [ ]:
# Plot all three scenarios
fig, ax = plt.subplots(figsize=(9, 5))
x_vals = np.linspace(5, 65, 300)
ax.plot(x_vals, R(x_vals),         "steelblue", lw=2, label="Baseline")
ax.plot(x_vals, R_climate(x_vals), "firebrick", lw=2, ls="--", label="Climate (−15%)")
ax.plot(x_vals, R_upgrade(x_vals), "green",     lw=2, ls=":",  label="Upgrade (+20%)")
ax.axvline(8,  color="orange", ls="--", alpha=0.5)
ax.axvline(60, color="orange", ls="--", alpha=0.5)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Panel Row Spacing x (metres)")
ax.set_ylabel("Annual Revenue ($000)")
ax.set_title("Solar Farm: Scenario Comparison")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

✏️ **CEO Recommendation (4–5 sentences):**
- Optimal spacing and expected revenue
- Climate risk: how much revenue could be lost?
- Whether the panel upgrade is financially justified

```
RECOMMENDATION:
...
```

---
## Part D: Symbolic Differentiation with SymPy (~10 min)

In Part A you found $R'(x)$ **by hand** using the power rule, then solved $R'(x)=0$ for the critical point. Python's **SymPy** library performs this kind of algebra automatically — the same **symbolic computation** idea you used on paper — and it's the standard tool for calculus in Python throughout this unit.

Run the cell below to have SymPy differentiate $R(x)$ symbolically and solve for the critical point, and confirm it matches your by-hand result from Part A.

In [ ]:
# D.1 — Symbolic differentiation with SymPy
import sympy as sp

x_sym = sp.symbols('x')
R_expr = -2*x_sym**2 + 180*x_sym - 1600

dR_expr = sp.diff(R_expr, x_sym)
print(f"R(x)  = {R_expr}")
print(f"R'(x) = {dR_expr}   ← same as the by-hand result from Part A")

# Solve R'(x) = 0 symbolically — same critical point as Part A.1
critical_points = sp.solve(sp.Eq(dR_expr, 0), x_sym)
print(f"\nCritical point(s): x = {critical_points}")

**Why this matters:** `sp.diff()` performs **symbolic differentiation** — it manipulates the algebraic expression using the same rules (power rule, etc.) you use on paper, rather than approximating a derivative from numbers the way `numpy` would. `sp.solve()` then finds the exact critical point algebraically.

---
## ✅ Presentation Checklist (Week 6, 10 minutes)

1. **Problem** (~2 min): Explain why panel row spacing affects revenue.
2. **Model** (~3 min): Present the revenue function, derivative, and critical point.
3. **Results** (~3 min): State optimal spacing, break-even range, and climate scenario outcomes.
4. **Recommendation** (~2 min): Give a spacing decision and upgrade advice with supporting numbers.

---
## 📊 Presentation Marking Rubric (20 marks → scaled to 4% of your unit grade)

Your group presentation is graded out of 20 marks (scaled to 4% of your unit grade — each group presents twice, for 8% total). This rubric determines your group mark, which will be the mark for each contributing member unless we are advised otherwise.

| Criterion | Excellent | Good | Developing | Poor |
|---|---|---|---|---|
| **Problem Formulation** (5 marks) | Clear explanation of the real-world problem; audience understands what question is being answered and why it matters (5) | Problem explained but lacks full context or motivation (3–4) | Problem stated but unclear why it's important (1–2) | No clear problem statement (0) |
| **Mathematical Approach** (5 marks) | Correct model/method selected; clear justification for the choice; key equations presented clearly (5) | Correct approach with minor errors; justification present but weak (3–4) | Approach has errors or is poorly justified (1–2) | Wrong method or no mathematical content shown (0) |
| **Results & Interpretation** (6 marks) | Results are correct and clearly presented; findings are connected to a real-world decision, including limitations/trade-offs (6) | Results mostly correct; some interpretation but lacks depth (4–5) | Results unclear, minor errors, or interpretation is minimal — just states numbers (2–3) | Major errors, no results shown, or no interpretation given (0–1) |
| **Communication Quality** (4 marks) | Effective graphs/visuals support the story; all members participate and speak without reading from notes; well-rehearsed and within the 10-minute limit (4) | Visuals adequate; most members participate; slightly over/under time (3) | Visualization ineffective or missing; uneven participation; timing issues (1–2) | No visuals; one person dominates; major timing problems (0) |

**Total: ____ / 20 marks**